# Task 4.3: Gradient Harmonization (FedGH)
**Goal: Detect and resolve gradient conflicts at server before aggregation**

## Setup

In [ ]:
import torch
import numpy as np
import copy

from scriptsfl.data_utils import load_dataset, create_dirichlet_split, get_dataloaders
from scriptsfl.models import get_model
from scriptsfl.client import Client
from scriptsfl.federated_utils import get_model_weights, set_model_weights, flatten_weights
from scriptsfl.server import Server
from scriptsfl.results_utils import save_results, plot_comparison

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
print("="*80)
print("TASK 4.3: GRADIENT HARMONIZATION (FedGH)")
print("="*80)

#%% Configuration
CONFIG = {
    'dataset': 'cifar10',
    'num_clients': 5,
    'num_rounds': 50,
    'local_epochs': 5,
    'batch_size': 32,
    'lr': 0.01,
    'alpha': 0.1,  # High heterogeneity
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

#%% Load data (reuse scripts)
train_dataset, test_dataset = load_dataset(CONFIG['dataset'])
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False)

client_indices = create_dirichlet_split(train_dataset, CONFIG['num_clients'], alpha=CONFIG['alpha'])
client_loaders = get_dataloaders(train_dataset, client_indices, batch_size=CONFIG['batch_size'])

## Gradient Harmonization Function

In [ ]:
def harmonize_gradients(client_updates):
    """
    Harmonize conflicting gradients using projection

    Algorithm:
    1. For each pair of clients (i, j):
        - Compute cosine similarity of their updates
        - If similarity < 0 (conflict), project each onto orthogonal complement of the other

    Args:
        client_updates: list of [weight updates for each client]

    Returns:
        harmonized updates (list)
    """
    num_clients = len(client_updates)

    # TODO: Flatten updates to vectors for easier computation
    # Hint: Use flatten_weights from federated_utils

    flattened = [flatten_weights(update) for update in client_updates]

    # TODO: For each pair (i, j):
    for i in range(num_clients):
        for j in range(i + 1, num_clients):
            # TODO: Compute dot product and norms
            gi = flattened[i]
            gj = flattened[j]

            dot_product = torch.dot(gi, gj)

            # TODO: Check if conflict exists (dot_product < 0)
            if dot_product < 0:
                # TODO: Project gi onto orthogonal complement of gj
                # Formula: gi' = gi - (gi·gj / ||gj||^2) * gj

                # TODO: Project gj onto orthogonal complement of gi
                # Formula: gj' = gj - (gi·gj / ||gi||^2) * gi

                # Note: Update flattened[i] and flattened[j] in place
                pass

    # TODO: Unflatten back to original weight structure
    # Hint: Use unflatten_weights from federated_utils

    # Return harmonized updates
    return client_updates  # TODO: Return harmonized version


#%% FedGH Server

class FedGHServer(Server):
    """
    Server with Gradient Harmonization

    Applies harmonization before aggregation
    """

    def aggregate(self, clients):
        """
        Aggregate with gradient harmonization
        """
        # Get initial global weights
        global_weights = get_model_weights(self.global_model)

        # Get client weights
        client_weights = [client.get_weights() for client in clients]

        # TODO: Compute weight updates (deltas)
        # delta_i = theta_i - theta_global
        client_updates = []
        for cw in client_weights:
            update = [cw_param - gw_param for cw_param, gw_param in zip(cw, global_weights)]
            client_updates.append(update)

        # TODO: Apply gradient harmonization
        harmonized_updates = harmonize_gradients(client_updates)

        # TODO: Aggregate harmonized updates
        # theta_global_new = theta_global + weighted_avg(harmonized_updates)

        # TODO: Set new global model weights

        return 0.0  # TODO: Return divergence if needed






## Run Experiments


In [ ]:

print("\n--- Training with FedGH ---")

# TODO: Initialize model and clients (use regular Client class)
model = get_model('simplecnn', num_classes=10, dataset=CONFIG['dataset'])
clients = [Client(i, client_loaders[i], model, CONFIG['device'])
           for i in range(CONFIG['num_clients'])]

# TODO: Create FedGH server
server = FedGHServer(model, clients, test_loader, CONFIG['device'])

# TODO: Train
history = server.train(
    num_rounds=CONFIG['num_rounds'],
    local_epochs=CONFIG['local_epochs'],
    lr=CONFIG['lr'],
    client_fraction=1.0,
    verbose=True
)

# TODO: Compare with FedAvg baseline



In [ ]:
print("\n" + "="*80)
print("TASK 4.3 IMPLEMENTATION NOTES")
print("="*80)
print("FedGH Key Points:")
print("1. Server-side intervention (no client changes needed)")
print("2. Detects conflicting gradients (cosine similarity < 0)")
print("3. Projects conflicting gradients to remove opposing components")
print("4. O(M^2) complexity for M clients (pairwise comparisons)")
print("5. Works well with 5-10 clients, expensive for larger M")
print("\nExpected result: FedGH should improve convergence stability")
print("and boost accuracy by 2-5% in highly non-IID scenarios")